Цель эксперимента: Обучить и локально протестировать базовую модель градиентного бустинга LightGBM, корректно настроив обработку встроенных категориальных признаков.
Какие данные используются: Датасет cleaned_bnpl_data.csv.
Какие основные выводы: Успешно настроен пайплайн обучения с переводом строковых колонок в тип данных category. Обученная модель LightGBM показала высокую пропускную способность, сбалансированные метрики точности (Precision/Recall) и была экспортирована в файл production_model.pkl как основной кандидат для интеграции в REST API.

In [1]:
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
import os
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from pathlib import Path
import joblib

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)
print('Working dir:', Path.cwd())

if not os.path.exists('data/cleaned_bnpl_data.csv'):
    raise FileNotFoundError(
        "Файл 'data/cleaned_bnpl_data.csv' не найден! "
        "Сначала запусти в терминале скрипт обработки: python src/data/prepare_data.py"
    )

df_clean = pd.read_csv('data/cleaned_bnpl_data.csv')

if 'Target' not in df_clean.columns:
    if 'Repayment_Status' in df_clean.columns:
        df_clean['Target'] = df_clean['Repayment_Status'].isin(['Defaulted', 'Late Payment']).astype(int)
    else:
        raise KeyError('Колонки Target или Repayment_Status не найдены в датасете!')

drop_cols = ['Target']
if 'Repayment_Status' in df_clean.columns:
    drop_cols.append('Repayment_Status')

X = df_clean.drop(columns=drop_cols)
y = df_clean['Target']

categorical_features = [
    'Gender', 'Purchase_Category', 'BNPL_Provider', 
    'Device_Type', 'Connection_Type', 'Browser'
]

for col in categorical_features:
    if col in X.columns:
        X[col] = X[col].astype('category')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Размер обучающей выборки X_train: {X_train.shape}")
print(f"Распределение классов в y_train:\n{y_train.value_counts(normalize=True)}")

Working dir: c:\Users\fedor\Documents\proga\repository-fedi-gr1\project
Размер обучающей выборки X_train: (40000, 11)
Распределение классов в y_train:
Target
0    0.767975
1    0.232025
Name: proportion, dtype: float64


In [2]:

param_dist = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 8, -1],
    'learning_rate': [0.01, 0.05, 0.1],
    'num_leaves': [31, 63, 127]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = RandomizedSearchCV(
    lgb.LGBMClassifier(class_weight='balanced', random_state=42),
    param_distributions=param_dist,
    n_iter=10,
    scoring='recall',
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=2,
    refit=True
)

print("Запускаю подбор гиперпараметров LightGBM...")
search.fit(X_train, y_train)
model = search.best_estimator_
print('Лучшие параметры:', search.best_params_)

Запускаю подбор гиперпараметров LightGBM...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
[LightGBM] [Info] Number of positive: 9281, number of negative: 30719
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001485 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1015
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fur

In [4]:
model = search.best_estimator_

y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred_class = model.predict(X_test)

auc = roc_auc_score(y_test, y_pred_proba)
print(f"\nROC-AUC Score на тесте: {auc:.4f}")
print("\nОтчет классификации:")
print(classification_report(y_test, y_pred_class, target_names=['Paid On Time', 'Default/Late']))

os.makedirs('artifacts/models', exist_ok=True)
model_path = 'artifacts/models/lgbm_risk_model.pkl'

joblib.dump(model, model_path)
print(f"\nМодель успешно сохранена в: {os.path.abspath(model_path)}")


ROC-AUC Score на тесте: 0.7145

Отчет классификации:
              precision    recall  f1-score   support

Paid On Time       0.90      0.56      0.69      7680
Default/Late       0.35      0.80      0.49      2320

    accuracy                           0.62     10000
   macro avg       0.63      0.68      0.59     10000
weighted avg       0.77      0.62      0.64     10000


Модель успешно сохранена в: c:\Users\fedor\Documents\proga\repository-fedi-gr1\project\artifacts\models\lgbm_risk_model.pkl
